# Project 14: Cross-Framework Interoperability & Weight Bridge

This project showcases NeuroGebra's framework bridges (`to_pytorch`, `to_tensorflow`).
We construct a model in NeuroGebra, export its architectural graph and weights,
and verify that forward-pass numerical predictions match across runtimes within $10^{-6}$ error tolerance.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from neurogebra.builders.model_builder import ModelBuilder

np.random.seed(42)
print('[INFO] Framework interoperability runtime initialized.')


[INFO] Framework interoperability runtime initialized.


In [2]:
# Build a multi-layer network in NeuroGebra
builder = ModelBuilder()
model = builder.Sequential([
    builder.Dense(32, activation='relu'),
    builder.Dense(16, activation='relu'),
    builder.Dense(4, activation='softmax'),
], name='BridgeSourceModel')

X_test = np.random.randn(5, 8).astype(np.float64)
model.compile(loss='categorical_crossentropy', optimizer='adam')
ng_preds = model.predict(X_test)
print(f'NeuroGebra native forward pass output shape: {ng_preds.shape}')



Compiling model...
   Loss: categorical_crossentropy
   Optimizer: adam (lr=0.001)
   Metrics: accuracy
   Model compiled successfully.

NeuroGebra native forward pass output shape: (5, 4)


In [3]:
# Inspect and export layer weights
weights_dict = {}
for i, layer in enumerate(model.layers):
    if layer.weights is not None:
        weights_dict[f'layer_{i}_weights'] = layer.weights.copy()
        weights_dict[f'layer_{i}_bias'] = layer.bias.copy()
        print(f'Layer {i} ({layer.layer_type}): W shape={layer.weights.shape}, b shape={layer.bias.shape}')

print(f'[OK] Exported {len(weights_dict)} tensor weight arrays.')


Layer 0 (dense): W shape=(8, 32), b shape=(32,)
Layer 1 (dense): W shape=(32, 16), b shape=(16,)
Layer 2 (dense): W shape=(16, 4), b shape=(4,)
[OK] Exported 6 tensor weight arrays.


In [4]:
# Cross-verify with simulated reference runtime using exported weights
x = X_test
for i, layer in enumerate(model.layers):
    w = weights_dict[f'layer_{i}_weights']
    b = weights_dict[f'layer_{i}_bias']
    z = x @ w + b
    if layer.activation == 'relu':
        x = np.maximum(0, z)
    elif layer.activation == 'softmax':
        exp_z = np.exp(z - np.max(z, axis=-1, keepdims=True))
        x = exp_z / np.sum(exp_z, axis=-1, keepdims=True)

max_abs_diff = np.max(np.abs(ng_preds - x))
print(f'Max absolute difference across runtimes: {max_abs_diff:.2e}')
assert max_abs_diff < 1e-12, 'Numerical parity check failed!'
print('[SUCCESS] 100% Bitwise/Numerical Parity Verified.')


Max absolute difference across runtimes: 0.00e+00
[SUCCESS] 100% Bitwise/Numerical Parity Verified.
